# 01 – Deploy Infrastructure

This notebook creates the Azure resource group and Azure OpenAI account with a model deployment using the **Azure CLI**.

**Estimated time:** 10–15 minutes (most of that is Azure provisioning)

---

## What gets deployed

- Azure Resource Group (if it does not exist)
- Azure OpenAI account (`Microsoft.CognitiveServices/accounts`, kind `OpenAI`) in **Sweden Central**
- Model deployment: `gpt-4o-mini` (configurable)

Infrastructure is defined in `infra/main.bicep`.

---

## Step 1 – Set deployment variables

Edit the values in this cell. The account name must be globally unique (3–24 lowercase alphanumeric).

In [ ]:
RESOURCE_GROUP  = "rg-foundry-feasibility"      # ← change if needed
LOCATION        = "swedencentral"
ACCOUNT_NAME    = "aoai-foundry-feasibility"    # ← must be globally unique
MODEL_NAME      = "gpt-4o-mini"
MODEL_VERSION   = "2024-07-18"
DEPLOYMENT_NAME = "chat"

print("Variables set:")
print(f"  Resource group : {RESOURCE_GROUP}")
print(f"  Location       : {LOCATION}")
print(f"  Account name   : {ACCOUNT_NAME}")
print(f"  Model          : {MODEL_NAME} ({MODEL_VERSION})")
print(f"  Deployment     : {DEPLOYMENT_NAME}")

## Step 2 – Create the resource group

In [ ]:
!az group create \
    --name {RESOURCE_GROUP} \
    --location {LOCATION} \
    --output table

## Step 3 – Deploy the Bicep template

This runs `infra/main.bicep` and passes the variables defined above.

> Provisioning typically takes 3–5 minutes.

In [ ]:
!az deployment group create \
    --resource-group {RESOURCE_GROUP} \
    --template-file ../infra/main.bicep \
    --parameters \
        location={LOCATION} \
        accountName={ACCOUNT_NAME} \
        modelName={MODEL_NAME} \
        modelVersion={MODEL_VERSION} \
        deploymentName={DEPLOYMENT_NAME} \
    --output table

## Step 4 – Retrieve the endpoint and API key

Once provisioning completes, collect the values you need for the `.env` file.

In [ ]:
import subprocess, json

# Retrieve the endpoint
result = subprocess.run(
    ["az", "cognitiveservices", "account", "show",
     "--name", ACCOUNT_NAME,
     "--resource-group", RESOURCE_GROUP,
     "--query", "properties.endpoint",
     "--output", "tsv"],
    capture_output=True, text=True
)
endpoint = result.stdout.strip()

# Retrieve the API key
key_result = subprocess.run(
    ["az", "cognitiveservices", "account", "keys", "list",
     "--name", ACCOUNT_NAME,
     "--resource-group", RESOURCE_GROUP,
     "--query", "key1",
     "--output", "tsv"],
    capture_output=True, text=True
)
api_key = key_result.stdout.strip()

print(f"Endpoint       : {endpoint}")
print(f"API Key        : {api_key[:8]}... (truncated for safety)")
print(f"Deployment     : {DEPLOYMENT_NAME}")

## Step 5 – Write values to `.env`

This cell writes the retrieved values to `env/.env` so the next notebook can load them automatically.

> **Security note:** `.env` is git-ignored. Never commit real keys.

In [ ]:
import pathlib

env_content = f"""# Auto-generated by 01_deploy_infra.ipynb – do not commit
AZURE_OPENAI_ENDPOINT={endpoint}
AZURE_OPENAI_API_KEY={api_key}
AZURE_OPENAI_DEPLOYMENT={DEPLOYMENT_NAME}
AZURE_OPENAI_API_VERSION=2024-10-21
"""

env_path = pathlib.Path("../env/.env")
env_path.write_text(env_content)
print(f"✅  Written to {env_path.resolve()}")

---

## ✅ Infrastructure deployed

Your Azure OpenAI account and model are live. Move to **`02_prompt_test.ipynb`** to run prompts against the deployment.

---

## Tear-down (optional)

When you have finished with the demo, delete the resource group to stop all charges.

In [ ]:
# Uncomment and run to delete all resources
# !az group delete --name {RESOURCE_GROUP} --yes --no-wait
# print("Resource group deletion triggered.")